In [56]:
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json

In [2]:
BASE_URL = "https://api.imdbapi.dev/"

In [64]:
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2025")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 177774}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25339}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48040}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 16764}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4210}",High college students face an unexpectedly epi...
5,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025,7860.0,"[Drama, Mystery, Thriller]","{'aggregateRating': 6.8, 'voteCount': 128732}",A struggling young woman is relieved by the ch...
6,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 52350}",A group of friends are going through a mid-lif...
7,tt27552099,movie,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,{'url': 'https://m.media-amazon.com/images/M/M...,2026,6420.0,"[Action, Comedy, Crime]","{'aggregateRating': 6.2, 'voteCount': 12170}",Two friends navigate the dangerous world of or...
8,tt39139925,movie,Dhurandhar The Revenge,Dhurandhar The Revenge,{'url': 'https://m.media-amazon.com/images/M/M...,2026,13800.0,"[Action, Crime, Thriller]","{'aggregateRating': 8.5, 'voteCount': 43679}",Jaskirat Singh Rangi descends deeper into his ...
9,tt0049833,movie,The Ten Commandments,The Ten Commandments,{'url': 'https://m.media-amazon.com/images/M/M...,1956,13200.0,"[Adventure, Drama, Family, History]","{'aggregateRating': 7.9, 'voteCount': 84366}","Moses, raised as a prince of Egypt in the Phar..."


In [65]:
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2024")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
print(f"Page 0: Fetched initial {len(df)} rows")
time.sleep(1)  

for page in range(1, 20):
      try:
            response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear=2026&pageToken={pagetoken}", timeout=10)
            
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Page {page}: Rate limited! Waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            
            if not response.text:
                  print(f"Page {page}: Empty response - reached end of data")
                  break
            
            data = response.json()
            
            if "titles" not in data or not data["titles"]:
                  print(f"Page {page}: No more titles - reached end of data")
                  break
                  
            df_page = pd.DataFrame(data["titles"])
            df = pd.concat([df, df_page], ignore_index=True)
            
            if "nextPageToken" in data and data["nextPageToken"]:
                  pagetoken = data["nextPageToken"]
            else:
                  print(f"Page {page}: No nextPageToken - reached end")
                  break
            
            time.sleep(1)  
      except Exception as e:
            print(f"Page {page}: Error - {type(e).__name__}: {e}")
            break

print(f"\n✓ Final shape: {df.shape}")
print(f"✓ Total rows: {len(df)}")
display(df.head())

Page 0: Fetched initial 50 rows

✓ Final shape: (1000, 10)
✓ Total rows: 1000


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 177774}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25339}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48040}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 16764}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4210}",High college students face an unexpectedly epi...


In [58]:
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 177774}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25339}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48040}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 16764}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4210}",High college students face an unexpectedly epi...
...,...,...,...,...,...,...,...,...,...,...
995,tt36933185,movie,Dead Howling,Dead Howling,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,NaN,[Horror],NaN,A werewolf hunts zombies after losing his fami...
996,tt39654006,movie,Cornbread Mafia,Cornbread Mafia,NaN,2026.0,4980.0,"[Documentary, Crime]","{'aggregateRating': 7.8, 'voteCount': 8}",Explores Kentucky farmers who created America'...
997,tt32140499,movie,The Old Blue Eyes,The Old Blue Eyes,{'url': 'https://m.media-amazon.com/images/M/M...,NaN,NaN,"[Biography, Drama, History, Romance]",NaN,NaN
998,tt40729378,movie,Flytrapper,Flytrapper,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,4440.0,[Horror],NaN,Crystal's life of extremely aggressive rap mus...


In [66]:
def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)  #
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        return ind, worldwide_gross, production_budget, None
    except requests.Timeout:
        return ind, None, None, f"{title_id}: Timeout"
    except requests.HTTPError as e:
        return ind, None, None, f"{title_id}: HTTP {response.status_code}"
    except json.JSONDecodeError:
        return ind, None, None, f"{title_id}: Invalid JSON response"
    except Exception as e:
        return ind, None, None, f"{title_id}: {type(e).__name__}"

skipped_count = 0
for ind, row in df.iterrows():
    ind_result, worldwide_gross, production_budget, error = fetch_box_office(ind, row, BASE_URL)
    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "productionBudget"] = production_budget
    
    if (ind + 1) % 100 == 0:
        print(f"Processed {ind + 1}/{len(df)} titles...")

print(f"\n✓ Complete! Skipped {skipped_count} titles due to errors")
print(f"✓ Box office data added to {len(df) - skipped_count} rows")

Processed 100/1000 titles...
Processed 200/1000 titles...
Processed 300/1000 titles...
Processed 400/1000 titles...
Processed 500/1000 titles...
Processed 600/1000 titles...
Processed 700/1000 titles...
Processed 800/1000 titles...
Processed 900/1000 titles...
Processed 1000/1000 titles...

✓ Complete! Skipped 0 titles due to errors
✓ Box office data added to 1000 rows


In [69]:
df.to_csv("data/box_office_data_n1000.csv", index=False)

In [67]:
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,worldwideGross,productionBudget
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 177774}",A science teacher wakes up alone on a spaceshi...,433030505,200000000
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25339}","Mario ventures into space, exploring cosmic wo...",437751829,110000000
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48040}","An elusive thief, eyeing his final score, enco...",72559167,90000000
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 16764}",A happily engaged couple is put to the test wh...,26232083,None
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4210}",High college students face an unexpectedly epi...,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
995,tt0374900,movie,Napoleon Dynamite,Napoleon Dynamite,{'url': 'https://m.media-amazon.com/images/M/M...,2004.0,5760.0,[Comedy],"{'aggregateRating': 7, 'voteCount': 255414}",A listless and alienated teenager helps his ne...,46141106,400000
996,tt0095765,movie,Cinema Paradiso,Nuovo Cinema Paradiso,{'url': 'https://m.media-amazon.com/images/M/M...,1988.0,10440.0,"[Drama, Romance]","{'aggregateRating': 8.5, 'voteCount': 316658}","Salvatore, a famous film director, returns to ...",13502484,5000000
997,tt1226837,movie,Beautiful Boy,Beautiful Boy,{'url': 'https://m.media-amazon.com/images/M/M...,2018.0,7200.0,"[Biography, Drama]","{'aggregateRating': 7.4, 'voteCount': 127712}",A family coping with addiction over many years...,31749905,25000000
998,tt0108399,movie,True Romance,True Romance,{'url': 'https://m.media-amazon.com/images/M/M...,1993.0,7140.0,"[Crime, Drama, Romance, Thriller]","{'aggregateRating': 7.9, 'voteCount': 260376}","In Detroit, a pop-culture enthusiast steals co...",13097916,13000000


In [68]:
df = df[(df["productionBudget"].notnull()) & (df["worldwideGross"].notnull())]
print(len(df))

742


In [6]:
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,endYear,worldwideGross,productionBudget
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026,9360.0,"['Drama', 'Sci-Fi', 'Thriller']","{'aggregateRating': 8.4, 'voteCount': 127808}",A science teacher wakes up alone on a spaceshi...,NaN,300802240.0,200000000.0
12,tt8036976,movie,Send Help,Send Help,{'url': 'https://m.media-amazon.com/images/M/M...,2026,6780.0,"['Adventure', 'Horror', 'Thriller']","{'aggregateRating': 6.9, 'voteCount': 57132}",When an employee and her insufferable boss bec...,NaN,93926663.0,40000000.0
16,tt30144839,movie,One Battle After Another,One Battle After Another,{'url': 'https://m.media-amazon.com/images/M/M...,2025,9660.0,"['Action', 'Crime', 'Drama', 'Thriller']","{'aggregateRating': 7.7, 'voteCount': 377690}","When their enemy resurfaces after 16 years, a ...",NaN,210906028.0,130000000.0
20,tt33978029,movie,Ready or Not 2: Here I Come,Ready or Not 2: Here I Come,{'url': 'https://m.media-amazon.com/images/M/M...,2026,6480.0,"['Comedy', 'Horror', 'Thriller']","{'aggregateRating': 7, 'voteCount': 9330}","After surviving one deadly game, Grace and her...",NaN,24038575.0,20000000.0
21,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025,5940.0,"['Action', 'Adventure', 'Comedy']","{'aggregateRating': 5.6, 'voteCount': 45137}",A group of friends are going through a mid-lif...,NaN,134974943.0,45000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,tt0119217,movie,Good Will Hunting,Good Will Hunting,{'url': 'https://m.media-amazon.com/images/M/M...,1997,7560.0,"['Drama', 'Romance']","{'aggregateRating': 8.4, 'voteCount': 1209614}","A therapist counsels Will Hunting, a janitor w...",NaN,225933435.0,10000000.0
484,tt0266697,movie,Kill Bill: Vol. 1,Kill Bill: Vol. 1,{'url': 'https://m.media-amazon.com/images/M/M...,2003,6660.0,"['Action', 'Crime', 'Thriller']","{'aggregateRating': 8.2, 'voteCount': 1301574}","After waking from a four-year coma, a former a...",NaN,180908413.0,30000000.0
493,tt20969586,movie,Thunderbolts*,Thunderbolts*,{'url': 'https://m.media-amazon.com/images/M/M...,2025,7620.0,"['Action', 'Adventure', 'Crime', 'Drama', 'Fan...","{'aggregateRating': 7.1, 'voteCount': 271865}",After finding themselves ensnared in a death t...,NaN,382436917.0,180000000.0
497,tt5537002,movie,Killers of the Flower Moon,Killers of the Flower Moon,{'url': 'https://m.media-amazon.com/images/M/M...,2023,12360.0,"['Crime', 'Drama', 'History', 'Western']","{'aggregateRating': 7.5, 'voteCount': 305196}",When oil is discovered in 1920s Oklahoma under...,NaN,158772599.0,200000000.0


In [34]:
def check_starmeter(base_url=BASE_URL, max_pages=100, delay=1.0):
      response = requests.get(f"{base_url}/chart/starmeter", timeout=10)
      response.raise_for_status()
      data = response.json()
      items = []
      names = data.get("names", [])
      nextPageToken = data.get("nextPageToken")
      items.extend([{**name, "page": 0} for name in names])
      
      for page in range(1, max_pages):
            if not nextPageToken:
                  break
            print(f"Checking starmeter page {page}...")
            response = requests.get(f"{base_url}/chart/starmeter?pageToken={nextPageToken}", timeout=10)
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Rate limited on starmeter page {page}, waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            data = response.json()
            names = data.get("names", [])
            nextPageToken = data.get("nextPageToken")
            if not names:
                  print(f"No names found on starmeter page {page}, stopping.")
                  break
            items.extend([{**name, "page": page} for name in names])
            time.sleep(delay)
      
      return pd.DataFrame(items)

starmeter_df = check_starmeter()
starmeter_df.head()

Checking starmeter page 1...
Checking starmeter page 2...
Checking starmeter page 3...
Checking starmeter page 4...
Checking starmeter page 5...
Checking starmeter page 6...
Checking starmeter page 7...
Checking starmeter page 8...


ReadTimeout: HTTPSConnectionPool(host='api.imdbapi.dev', port=443): Read timed out. (read timeout=10)

In [ ]:
starmeter_df.to_csv("data/starmeter_top.csv", index=False)

In [40]:
starmeter_df = pd.read_csv("data/starmeter_top.csv")
starmeter_df

,id,displayName,primaryImage,heightCm,birthDate,deathDate,meterRanking,page
0,nm0281619,Carrie Anne Fleming,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1974, 'month': 8, 'day': 16}","{'year': 2026, 'month': 2, 'day': 26}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0
1,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}",NaN,"{'currentRank': 2, 'changeDirection': 'UP', 'd...",0
2,nm1683768,Beau Garrett,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1982, 'month': 12, 'day': 28}",NaN,"{'currentRank': 3, 'changeDirection': 'UP', 'd...",0
3,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}",NaN,"{'currentRank': 4, 'changeDirection': 'UP', 'd...",0
4,nm4422686,Barry Keoghan,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1992, 'month': 10, 'day': 18}",NaN,"{'currentRank': 5, 'changeDirection': 'UP', 'd...",0
...,...,...,...,...,...,...,...,...
9995,nm0930570,Evan Williams,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1985, 'month': 3, 'day': 11}",NaN,NaN,99
9996,nm2965271,Arthur Darvill,{'url': 'https://m.media-amazon.com/images/M/M...,180.0,"{'year': 1982, 'month': 6, 'day': 17}",NaN,NaN,99
9997,nm7669972,Maria Chiara Giannetta,{'url': 'https://m.media-amazon.com/images/M/M...,163.0,"{'year': 1992, 'month': 5, 'day': 20}",NaN,NaN,99
9998,nm0005254,Tahj Mowry,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1986, 'month': 5, 'day': 17}",NaN,NaN,99


In [41]:
import ast

# Parse the meterRanking dict and extract currentRank
starmeter_df["currentRank"] = starmeter_df["meterRanking"].apply(
    lambda x: ast.literal_eval(x).get("currentRank") if pd.notna(x) else None
)

# Bucket: rank 1-100 → tier 1, 101-200 → tier 2, etc.
starmeter_df["rankTier"] = starmeter_df["currentRank"].apply(
    lambda x: ((int(x) - 1) // 100) + 1 if pd.notna(x) else None
)

starmeter_df.head()

,id,displayName,primaryImage,heightCm,birthDate,deathDate,meterRanking,page,currentRank,rankTier
0,nm0281619,Carrie Anne Fleming,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1974, 'month': 8, 'day': 16}","{'year': 2026, 'month': 2, 'day': 26}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0,1.0,1.0
1,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}",NaN,"{'currentRank': 2, 'changeDirection': 'UP', 'd...",0,2.0,1.0
2,nm1683768,Beau Garrett,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1982, 'month': 12, 'day': 28}",NaN,"{'currentRank': 3, 'changeDirection': 'UP', 'd...",0,3.0,1.0
3,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}",NaN,"{'currentRank': 4, 'changeDirection': 'UP', 'd...",0,4.0,1.0
4,nm4422686,Barry Keoghan,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1992, 'month': 10, 'day': 18}",NaN,"{'currentRank': 5, 'changeDirection': 'UP', 'd...",0,5.0,1.0


In [ ]:
for ind, row in df.iterrows():
    stars = row.get("stars", [])
    if isinstance(stars, str):
        try:
            stars = ast.literal_eval(stars)
        except:
            stars = []
    
    total_star_meter = 0
    for star in stars:
        star_name = star.get("name")
        starmeter_row = starmeter_df[starmeter_df["name"] == star_name]
        if not starmeter_row.empty:
            rank_tier = starmeter_row.iloc[0]["rankTier"]
            if pd.notna(rank_tier):
                total_star_meter += 1 / rank_tier
    
    df.at[ind, "totalStarMeter"] = total_star_meter

,id,displayName,primaryImage,heightCm,birthDate,deathDate,meterRanking,page,currentRank,rankTier
0,nm0281619,Carrie Anne Fleming,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1974, 'month': 8, 'day': 16}","{'year': 2026, 'month': 2, 'day': 26}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0,1.0,1.0
1,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}",NaN,"{'currentRank': 2, 'changeDirection': 'UP', 'd...",0,2.0,1.0
2,nm1683768,Beau Garrett,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1982, 'month': 12, 'day': 28}",NaN,"{'currentRank': 3, 'changeDirection': 'UP', 'd...",0,3.0,1.0
3,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}",NaN,"{'currentRank': 4, 'changeDirection': 'UP', 'd...",0,4.0,1.0
4,nm4422686,Barry Keoghan,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1992, 'month': 10, 'day': 18}",NaN,"{'currentRank': 5, 'changeDirection': 'UP', 'd...",0,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...
9995,nm0930570,Evan Williams,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1985, 'month': 3, 'day': 11}",NaN,NaN,99,NaN,NaN
9996,nm2965271,Arthur Darvill,{'url': 'https://m.media-amazon.com/images/M/M...,180.0,"{'year': 1982, 'month': 6, 'day': 17}",NaN,NaN,99,NaN,NaN
9997,nm7669972,Maria Chiara Giannetta,{'url': 'https://m.media-amazon.com/images/M/M...,163.0,"{'year': 1992, 'month': 5, 'day': 20}",NaN,NaN,99,NaN,NaN
9998,nm0005254,Tahj Mowry,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1986, 'month': 5, 'day': 17}",NaN,NaN,99,NaN,NaN


In [51]:
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,endYear,worldwideGross,productionBudget
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Drama, Sci-Fi, Thriller]","{'aggregateRating': 8.4, 'voteCount': 129068}",A science teacher wakes up alone on a spaceshi...,NaN,300802240,200000000
12,tt8036976,movie,Send Help,Send Help,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6780.0,"[Adventure, Horror, Thriller]","{'aggregateRating': 6.9, 'voteCount': 57393}",When an employee and her insufferable boss bec...,NaN,94004231,40000000
16,tt30144839,movie,One Battle After Another,One Battle After Another,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,9660.0,"[Action, Crime, Drama, Thriller]","{'aggregateRating': 7.7, 'voteCount': 377867}","When their enemy resurfaces after 16 years, a ...",NaN,210906028,130000000
20,tt33978029,movie,Ready or Not 2: Here I Come,Ready or Not 2: Here I Come,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6480.0,"[Comedy, Horror, Thriller]","{'aggregateRating': 7, 'voteCount': 9410}","After surviving one deadly game, Grace and her...",NaN,24038575,20000000
21,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 45201}",A group of friends are going through a mid-lif...,NaN,134974943,45000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
987,tt0102057,movie,Hook,Hook,{'url': 'https://m.media-amazon.com/images/M/M...,1991.0,8520.0,"[Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.8, 'voteCount': 290602}","When Captain James Hook kidnaps his children, ...",NaN,300854823,70000000
991,tt8115900,movie,The Bad Guys,The Bad Guys,{'url': 'https://m.media-amazon.com/images/M/M...,2022.0,6000.0,"[Animation, Adventure, Comedy, Crime, Family]","{'aggregateRating': 6.9, 'voteCount': 79303}","To avoid prison, a gang of notorious animal cr...",NaN,250387888,70000000
992,tt0089218,movie,The Goonies,The Goonies,{'url': 'https://m.media-amazon.com/images/M/M...,1985.0,6840.0,"[Adventure, Comedy, Family]","{'aggregateRating': 7.7, 'voteCount': 325511}",A group of young misfits called The Goonies di...,NaN,65642529,19000000
997,tt2084970,movie,The Imitation Game,The Imitation Game,{'url': 'https://m.media-amazon.com/images/M/M...,2014.0,6840.0,"[Biography, Drama, Thriller, War]","{'aggregateRating': 8, 'voteCount': 875353}","During World War II, the English mathematical ...",NaN,233555708,14000000


In [54]:
starmeter_df

,id,displayName,primaryImage,heightCm,birthDate,deathDate,meterRanking,page,currentRank,rankTier
0,nm0281619,Carrie Anne Fleming,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1974, 'month': 8, 'day': 16}","{'year': 2026, 'month': 2, 'day': 26}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0,1.0,1.0
1,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}",NaN,"{'currentRank': 2, 'changeDirection': 'UP', 'd...",0,2.0,1.0
2,nm1683768,Beau Garrett,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1982, 'month': 12, 'day': 28}",NaN,"{'currentRank': 3, 'changeDirection': 'UP', 'd...",0,3.0,1.0
3,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}",NaN,"{'currentRank': 4, 'changeDirection': 'UP', 'd...",0,4.0,1.0
4,nm4422686,Barry Keoghan,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1992, 'month': 10, 'day': 18}",NaN,"{'currentRank': 5, 'changeDirection': 'UP', 'd...",0,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...
9995,nm0930570,Evan Williams,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1985, 'month': 3, 'day': 11}",NaN,NaN,99,NaN,NaN
9996,nm2965271,Arthur Darvill,{'url': 'https://m.media-amazon.com/images/M/M...,180.0,"{'year': 1982, 'month': 6, 'day': 17}",NaN,NaN,99,NaN,NaN
9997,nm7669972,Maria Chiara Giannetta,{'url': 'https://m.media-amazon.com/images/M/M...,163.0,"{'year': 1992, 'month': 5, 'day': 20}",NaN,NaN,99,NaN,NaN
9998,nm0005254,Tahj Mowry,{'url': 'https://m.media-amazon.com/images/M/M...,170.0,"{'year': 1986, 'month': 5, 'day': 17}",NaN,NaN,99,NaN,NaN


In [ ]:
# Initialize totalStarMeter column if it doesn't exist
if "totalStarMeter" not in df.columns:
    df["totalStarMeter"] = 0.0

print(f"Starting actor-movie join for {len(df)} titles...")

In [ ]:
for ind, row in df.iterrows():
    try:
        response = requests.get(f"{BASE_URL}/titles/{row['id']}", timeout=5)
        
        # Handle rate limiting
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 5))
            print(f"Rate limited at row {ind}, waiting {wait_time}s...")
            time.sleep(wait_time)
            continue
        
        # Check for other errors
        if response.status_code != 200:
            print(f"Row {ind}: HTTP {response.status_code}")
            continue
        
        # Verify response has content
        if not response.text:
            print(f"Row {ind}: Empty response")
            continue
        
        data = response.json()
        stars = data.get("stars", [])
        
        if not stars:
            continue
        
        for star in stars:
            try:
                star_name = star.get("id")
                if not star_name:
                    continue
                starmeter_row = starmeter_df[starmeter_df["displayName"] == star_name]
                if not starmeter_row.empty:
                    rank_tier = starmeter_row.iloc[0]["rankTier"]
                    if pd.notna(rank_tier):
                        df.at[ind, "totalStarMeter"] = df.at[ind, "totalStarMeter"] + (1 / rank_tier)
            except Exception as e:
                print(f"Row {ind}, star {star.get('id', 'unknown')}: {type(e).__name__}")
                continue
        
        if (ind + 1) % 50 == 0:
            print(f"Processed {ind + 1} rows...")
            
    except requests.Timeout:
        print(f"Row {ind}: Request timeout")
    except requests.JSONDecodeError as e:
        print(f"Row {ind}: Invalid JSON - {e}")
    except Exception as e:
        print(f"Row {ind}: {type(e).__name__}: {e}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [37]:
missing_counts = df.isna().sum()
print(missing_counts)
missing_counts[missing_counts > 0]

id                    0
type                  0
primaryTitle          0
originalTitle         0
primaryImage          1
startYear             1
runtimeSeconds       51
genres                0
rating               21
plot                  1
endYear             377
worldwideGross      304
productionBudget    344
dtype: int64


primaryImage          1
startYear             1
runtimeSeconds       51
rating               21
plot                  1
endYear             377
worldwideGross      304
productionBudget    344
dtype: int64